# CC-WBT-AI — Engine Exploration

This notebook verifies that the CC-WBT computation engine can be accessed and driven directly from Python, without launching the web interface or the API.

The goal is to confirm that:
- the `ExcelFormulaProcessor` can be imported and instantiated
- the Rwanda scenario files are readable and navigable
- key financial outputs (LTS, Cash Flow, DSCR) can be extracted programmatically

This is the foundation for the sensitivity analysis and Monte Carlo simulation that follow in subsequent notebooks.

## 1. Setup

Add the backend folder to the Python path so we can import the computation engine directly.

In [6]:
import sys
import os

# Setup: add backend to path so we can import the computation engine directly
BACKEND_PATH = os.path.join(os.getcwd(), 'backend')
BACKEND_PATH = os.path.abspath(BACKEND_PATH)
sys.path.insert(0, BACKEND_PATH)

print("Backend path:", BACKEND_PATH)
print("Exists:", os.path.exists(BACKEND_PATH))

Backend path: c:\Users\agarr\OneDrive\Escritorio\IIT\CC-WBT-AI\backend
Exists: True


## 2. Load the computation engine

`ExcelFormulaProcessor` is the core engine of CC-WBT. It reads financial formulas from `formulas_map.json`, expands them across all country × model × fuel combinations, and writes the computed outputs back to the Excel workbooks.

Importing it directly — without FastAPI or Streamlit — is what allows us to run the engine programmatically in a notebook.

In [7]:
from excel_formula_engine import ExcelFormulaProcessor

processor = ExcelFormulaProcessor()
print("ExcelFormulaProcessor loaded successfully")
print("Type:", type(processor))

ExcelFormulaProcessor loaded successfully
Type: <class 'excel_formula_engine.ExcelFormulaProcessor'>


## 3. Verify Rwanda scenario files

The Rwanda scenario is the reference case used to validate CC-WBT against the NICCP model. It contains three scenarios (Baseline, CleanStep, Aligned) across two active fuel markets (Electricity & E-Cooking, LPG) over a 2023–2034 planning horizon.

We check that the key Excel files for CleanStep are accessible.

In [8]:
RWANDA_PATH = os.path.join(BACKEND_PATH, 'Rwanda')

files_of_interest = [
    'design-capital-CleanStep.xlsx',
    'financial-statements-CleanStep.xlsx',
    'technoeconomic-inputs-CleanStep.xlsx',
]

print("Rwanda path:", RWANDA_PATH)
print("Exists:", os.path.exists(RWANDA_PATH))
print()
for f in files_of_interest:
    full_path = os.path.join(RWANDA_PATH, f)
    print(f"{'✓' if os.path.exists(full_path) else '✗'} {f}")

Rwanda path: c:\Users\agarr\OneDrive\Escritorio\IIT\CC-WBT-AI\backend\Rwanda
Exists: True

✓ design-capital-CleanStep.xlsx
✓ financial-statements-CleanStep.xlsx
✓ technoeconomic-inputs-CleanStep.xlsx


## 4. Read capital structure inputs

The capital structure defines how the project is financed. For Rwanda CleanStep — Electricity & E-Cooking:

| Parameter | Value |
|-----------|-------|
| Equity | 20% at 16% cost |
| Grants | 50%, 8-year realisation |
| Debt | 30% at 8%, 6yr grace, 25yr amortisation |

These are the parameters we will vary in the sensitivity analysis.

In [9]:
import openpyxl

design_capital_path = os.path.join(RWANDA_PATH, 'design-capital-CleanStep.xlsx')
wb = openpyxl.load_workbook(design_capital_path, data_only=True)
ws = wb['Electricity & E-Cooking']

params = {}
for row in ws.iter_rows(max_col=3, values_only=True):
    if row[0] in ['1. Equity', 'Cost of Equity', '2. Grants', 'Years realisation',
                  'Cost of Debt', 'Grace period', 'Amortization period']:
        params[row[0]] = row[2]

print("Current capital structure — Electricity & E-Cooking:")
for k, v in params.items():
    print(f"  {k}: {v}")

Current capital structure — Electricity & E-Cooking:
  1. Equity: 20
  Cost of Equity: 16
  2. Grants: 50
  Years realisation: 8
  Cost of Debt: 8
  Grace period: 6
  Amortization period: 25


## 5. Read financial outputs

From `financial-statements-CleanStep.xlsx` we extract the four variables needed to compute our output metrics:

- **Long-Term Subsidies (LTS)** — public subsidy required each year to cover the viability gap
- **Operating Cash Flow** — cash generated by operations before debt service
- **Financial Expense** — interest payments on debt
- **Debt repayment** — principal amortisation

Financial Expense and Debt repayment are signed negative in the model.

In [10]:
fs_path = os.path.join(RWANDA_PATH, 'financial-statements-CleanStep.xlsx')
wb_fs = openpyxl.load_workbook(fs_path, data_only=True)
ws_fs = wb_fs['Electricity & E-Cooking']

rows_of_interest = ['Long term subsidies', 'Operating Cash Flow', 'Financial Expense', 'Debt repayment']
outputs = {}

for row in ws_fs.iter_rows(values_only=True):
    if row[0] in rows_of_interest and row[1] == '-':
        outputs[row[0]] = [v for v in row[3:] if v is not None]

print("Key financial outputs — Electricity & E-Cooking (2023-2034):")
for k, v in outputs.items():
    print(f"\n  {k}:")
    print(f"  {[round(x,1) if x else 0 for x in v]}")

Key financial outputs — Electricity & E-Cooking (2023-2034):

  Long term subsidies:
  [0, 0, 27.0, 78.1, 115.4, 147.5, 182.8, 210.4, 232.6, 253.7, 274.2, 252.7]

  Operating Cash Flow:
  [102.2, 135.9, 188.6, 251.6, 312.3, 370.5, 433.4, 501.0, 565.8, 627.9, 687.3, 720.7]

  Financial Expense:
  [0, 0, 0, 0, 0, -4.8, -16, -29.2, -41.6, -51, -57.2, -58.4]

  Debt repayment:
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -29.8]


## 6. Compute DSCR

The **Debt Service Coverage Ratio (DSCR)** measures whether the project generates enough cash to cover its debt obligations each year:

$$DSCR = \frac{\text{Operating Cash Flow}}{|\text{Financial Expense}| + |\text{Debt repayment}|}$$

A DSCR below 1.0 means the project cannot meet its debt service. Banks typically require a minimum of **1.2**.

During the grace period (years 1–6 for CleanStep) there is no debt service, so DSCR is not applicable.

We compute two versions:
- **DSCR** — including LTS in Operating Cash Flow
- **DSCR ex-subsidies** — excluding LTS, since public subsidies are not a reliable cash flow source from a project finance perspective

In [11]:
years = list(range(2023, 2035))
ocf = outputs['Operating Cash Flow']
fin_exp = outputs['Financial Expense']
debt_rep = outputs['Debt repayment']
lts = outputs['Long term subsidies']

def compute_dscr(cash_flows, fin_exp, debt_rep):
    result = []
    for i in range(len(years)):
        debt_service = abs(fin_exp[i]) + abs(debt_rep[i])
        if debt_service == 0:
            result.append(None)
        else:
            result.append(round(cash_flows[i] / debt_service, 2))
    return result

dscr = compute_dscr(ocf, fin_exp, debt_rep)
ocf_ex_sub = [round(ocf[i] - lts[i], 1) for i in range(len(years))]
dscr_ex_sub = compute_dscr(ocf_ex_sub, fin_exp, debt_rep)

def print_dscr(label, values):
    print(f"{label}:")
    for year, d in zip(years, values):
        if d is None:
            print(f"  {year}: — (grace period)")
        else:
            status = "✓" if d >= 1.2 else "⚠ BELOW 1.2"
            print(f"  {year}: {d} {status}")

print_dscr("DSCR — Electricity & E-Cooking (2023-2034)", dscr)
print()
print_dscr("DSCR ex-subsidies — Electricity & E-Cooking (2023-2034)", dscr_ex_sub)

DSCR — Electricity & E-Cooking (2023-2034):
  2023: — (grace period)
  2024: — (grace period)
  2025: — (grace period)
  2026: — (grace period)
  2027: — (grace period)
  2028: 77.18 ✓
  2029: 27.09 ✓
  2030: 17.16 ✓
  2031: 13.6 ✓
  2032: 12.31 ✓
  2033: 12.02 ✓
  2034: 8.17 ✓

DSCR ex-subsidies — Electricity & E-Cooking (2023-2034):
  2023: — (grace period)
  2024: — (grace period)
  2025: — (grace period)
  2026: — (grace period)
  2027: — (grace period)
  2028: 46.44 ✓
  2029: 15.66 ✓
  2030: 9.95 ✓
  2031: 8.01 ✓
  2032: 7.34 ✓
  2033: 7.22 ✓
  2034: 5.31 ✓


## Summary

The exploration confirms that the CC-WBT engine is fully accessible from Python:

- ✅ `ExcelFormulaProcessor` imports and instantiates correctly
- ✅ Rwanda scenario files are readable via `openpyxl`
- ✅ Capital structure inputs can be read from `design-capital-CleanStep.xlsx`
- ✅ Financial outputs (LTS, OCF, Financial Expense, Debt repayment) can be extracted from `financial-statements-CleanStep.xlsx`
- ✅ DSCR can be derived from the available outputs

The DSCR values for Rwanda CleanStep are high (5–46x depending on whether subsidies are included), which reflects the low relative debt burden of this scenario (30% of CAPEX, 6-year grace, 25-year amortisation). This will change significantly as we vary the capital structure parameters in the sensitivity analysis.

**Next:** `02_sensitivity_analysis.ipynb` — vary capital structure parameters one at a time and observe the impact on LTS, Cash Flow and DSCR.